In [0]:
CATALOG = "weather_project"
SCHEMA  = "default"
SILVER_TABLE     = f"{CATALOG}.{SCHEMA}.silver_weather_clean"
BRONZE_TABLE     = f"{CATALOG}.{SCHEMA}.bronze_weather_raw"
QUARANTINE_TABLE = f"{CATALOG}.{SCHEMA}.silver_weather_quarantine"



df = spark.read.table(SILVER_TABLE)

# Table 1  —  gold_city_weather_snapshot
# 
## Business question: What is the current weather condition in each city right now?

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import col, row_number, when

# Table names
GOLD_TABLE = f"{CATALOG}.{SCHEMA}.gold_city_weather_snapshot"

# Read Silver table
df = spark.read.table(SILVER_TABLE)

#  Remove rows with null critical values
df_filtered = df.filter(
    col("temperature").isNotNull() &
    col("humidity").isNotNull() &
    col("event_time").isNotNull()
)

#  Get latest record per city
window_spec = Window.partitionBy("city").orderBy(col("event_time").desc())

df_latest = df_filtered.withColumn(
    "rn",
    row_number().over(window_spec)
).filter(col("rn") == 1)


# Create risk_level
df_final = df_latest.withColumn(
    "risk_level",
    when(col("weather_severity_score").between(0, 1), "Low")
    .when(col("weather_severity_score").between(2, 3), "Medium")
    .when(col("weather_severity_score").between(4, 5), "High")
    .otherwise("Critical")
)

#  Select final columns
df_gold = df_final.select(
    col("city"),
    col("event_time").alias("latest_event_time"),
    col("temperature").alias("latest_temperature"),
    col("humidity").alias("latest_humidity"),
    col("weather_condition").alias("latest_weather_condition"),
    col("rainfall").alias("latest_rainfall"),
    col("weather_severity_score").alias("latest_weather_severity_score"),
    col("risk_level")
)

#  Write Gold table (overwrite snapshot)
df_gold.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(GOLD_TABLE)
print("✅ Gold table created successfully!")

✅ Gold table created successfully!


In [0]:
spark.read.table(GOLD_TABLE).display()

city,latest_event_time,latest_temperature,latest_humidity,latest_weather_condition,latest_rainfall,latest_weather_severity_score,risk_level
Bangalore,2026-03-07T23:00:00.000Z,28.6,72,Cloudy,0.2,0,Low
Chennai,2026-03-07T23:00:00.000Z,36.8,78,Partly Cloudy,14.4,0,Low
Delhi,2026-03-07T23:00:00.000Z,30.9,52,Cloudy,7.7,0,Low
Hyderabad,2026-03-07T23:00:00.000Z,26.6,51,Clear,3.8,0,Low
Mumbai,2026-03-07T23:00:00.000Z,29.9,75,Rainy,13.9,0,Low
Pune,2026-03-07T23:00:00.000Z,23.9,77,Thunderstorm,4.9,0,Low


# Table 2  —  gold_heatwave_alerts
## # Business question: Which cities crossed 40°C at least 3 times in the last 7 days?
## 

In [0]:
from pyspark.sql.functions import col, min, max, when, date_sub, lit, countDistinct, expr
from pyspark.sql.functions import max as spark_max

GOLD_ALERT_TABLE = f"{CATALOG}.{SCHEMA}.gold_heatwave_alerts"

#  Read Silver table
df = spark.read.table(SILVER_TABLE)

# Get latest event_date from data
max_date = df.agg(spark_max("event_date")).collect()[0][0]

print(f"Max event_date in data: {max_date}")

# Filter last 7 days based on max_date
df_7days = df.filter(
    col("event_date") >= date_sub(lit(max_date), 6)
)

#  Filter heatwave condition (>= 40)
df_heatwave = df_7days.filter(col("temperature") >= 40)

# Aggregate per city (COUNT DISTINCT DAYS)
df_agg = df_heatwave.groupBy("city").agg(
    countDistinct("event_date").alias("heatwave_days"),
    min("event_time").alias("first_occurrence"),
    max("event_time").alias("latest_occurrence")
)

# Keep cities with >= 3 heatwave days
df_filtered = df_agg.filter(col("heatwave_days") >= 3)

#  Add alert status (based on recency)
df_final = df_filtered.withColumn(
    "alert_status",
    when(
        col("latest_occurrence") >= expr("timestampadd(DAY, -1, cast('{}' as timestamp))".format(max_date)),
        "Active"
    ).otherwise("Monitoring")
)

#  Select final columns
df_gold_alert = df_final.select(
    col("city"),
    col("heatwave_days"),
    col("first_occurrence"),
    col("latest_occurrence"),
    col("alert_status")
)

#Write Gold table
df_gold_alert.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(GOLD_ALERT_TABLE)

print("Gold heatwave alerts table created successfully!")

Max event_date in data: 2026-03-08
🔥 Gold heatwave alerts table created successfully!


In [0]:
spark.read.table(GOLD_ALERT_TABLE).display()

city,heatwave_days,first_occurrence,latest_occurrence,alert_status
Delhi,6,2026-03-02T04:00:00.000Z,2026-03-07T15:00:00.000Z,Active


# Table 3  —  gold_rainfall_streaks
## # Business question: Which city had the longest uninterrupted streak of non-zero rainfall?
## 

In [0]:
from pyspark.sql.functions import col, lag, when, sum as spark_sum, min, max, count, expr,row_number
from pyspark.sql.window import Window

GOLD_RAIN_TABLE = f"{CATALOG}.{SCHEMA}.gold_rainfall_streaks"

#  Read Silver table
df = spark.read.table(SILVER_TABLE)

#Keep only rainy hours
df_rain = df.filter(col("rainfall") > 0)

# Define window (ordered by time)
window_spec = Window.partitionBy("city").orderBy("event_time")

#Get previous timestamp
df_seq = df_rain.withColumn(
    "prev_time",
    lag("event_time").over(window_spec)
)

# Identify new streak (gap detection)
df_seq = df_seq.withColumn(
    "new_group",
    when(
        col("prev_time").isNull() |
        (col("event_time") != col("prev_time") + expr("INTERVAL 1 HOUR")),
        1
    ).otherwise(0)
)

 #Assign group_id (cumulative sum)
df_seq = df_seq.withColumn(
    "group_id",
    spark_sum("new_group").over(window_spec)
)

# Aggregate streaks
df_streaks = df_seq.groupBy("city", "group_id").agg(
    min("event_time").alias("streak_start"),
    max("event_time").alias("streak_end"),
    countDistinct("event_time").alias("streak_hours")
)

# Get longest streak per city
window_max = Window.partitionBy("city").orderBy(col("streak_hours").desc())

df_final = df_streaks.withColumn(
    "rn",
    row_number().over(window_max)
).filter(col("rn") == 1)

# Select final columns
df_gold_rain = df_final.select(
    "city",
    "streak_start",
    "streak_end",
    "streak_hours"
)

#  Write Gold table
df_gold_rain.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(GOLD_RAIN_TABLE)

print("🌧️ Gold rainfall streaks table created successfully!")

🌧️ Gold rainfall streaks table created successfully!


In [0]:
spark.read.table(GOLD_RAIN_TABLE).display()

city,streak_start,streak_end,streak_hours
Bangalore,2026-03-01T00:00:00.000Z,2026-03-04T23:00:00.000Z,96
Chennai,2026-03-01T00:00:00.000Z,2026-03-04T23:00:00.000Z,96
Delhi,2026-03-01T00:00:00.000Z,2026-03-05T00:00:00.000Z,97
Hyderabad,2026-03-01T00:00:00.000Z,2026-03-05T00:00:00.000Z,97
Mumbai,2026-03-02T13:00:00.000Z,2026-03-04T23:00:00.000Z,59
Pune,2026-03-01T00:00:00.000Z,2026-03-04T23:00:00.000Z,96


# Table 4  —  gold_temperature_change
## # Business question: Which cities had a temperature swing greater than 8°C within 6 hours?
## 

In [0]:
from pyspark.sql.functions import col, lag, abs as spark_abs
from pyspark.sql.window import Window

GOLD_TEMP_CHANGE_TABLE = f"{CATALOG}.{SCHEMA}.gold_temperature_change"

#  Read Silver
df = spark.read.table(SILVER_TABLE)

#  Window
window_spec = Window.partitionBy("city").orderBy("event_time")

#  Use 6-hour lag (CRITICAL FIX )
df_lag = df.withColumn(
    "previous_temperature",
    lag("temperature", 6).over(window_spec)
).withColumn(
    "previous_event_time",
    lag("event_time", 6).over(window_spec)
)

#  Remove nulls (first 6 rows per city)
df_lag = df_lag.filter(col("previous_temperature").isNotNull())

#  Calculate difference
df_final = df_lag.withColumn(
    "difference",
    spark_abs(col("temperature") - col("previous_temperature"))
).filter(col("difference") > 8)

#  Select columns
df_gold_temp = df_final.select(
    "city",
    "event_time",
    "previous_event_time",
    "previous_temperature",
    col("temperature").alias("current_temperature"),
    "difference"
)

# Write table
df_gold_temp.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(GOLD_TEMP_CHANGE_TABLE)

print(" Temperature change table created successfully!")

🌡️ Temperature change table created successfully!


In [0]:
spark.read.table(GOLD_TEMP_CHANGE_TABLE).display()

city,event_time,previous_event_time,previous_temperature,current_temperature,difference
Bangalore,2026-03-01T14:00:00.000Z,2026-03-01T08:00:00.000Z,32.0,23.2,8.8
Bangalore,2026-03-01T23:00:00.000Z,2026-03-01T17:00:00.000Z,20.0,30.2,10.2
Bangalore,2026-03-02T06:00:00.000Z,2026-03-02T00:00:00.000Z,22.3,31.0,8.7
Bangalore,2026-03-02T12:00:00.000Z,2026-03-02T06:00:00.000Z,31.0,22.8,8.2
Bangalore,2026-03-02T23:00:00.000Z,2026-03-02T17:00:00.000Z,21.7,30.9,9.2
Bangalore,2026-03-03T05:00:00.000Z,2026-03-02T23:00:00.000Z,30.9,22.5,8.399999999999999
Bangalore,2026-03-03T07:00:00.000Z,2026-03-03T01:00:00.000Z,21.9,31.0,9.100000000000001
Bangalore,2026-03-03T15:00:00.000Z,2026-03-03T09:00:00.000Z,30.2,20.5,9.7
Bangalore,2026-03-03T18:00:00.000Z,2026-03-03T12:00:00.000Z,21.5,30.4,8.899999999999999
Bangalore,2026-03-04T02:00:00.000Z,2026-03-03T20:00:00.000Z,21.0,29.8,8.8


# Table 5  —  gold_provider_quality
## # Business question: Which provider sends the most unreliable data?
## 

In [0]:
from pyspark.sql.functions import col, count, round as spark_round

GOLD_PROVIDER_TABLE = f"{CATALOG}.{SCHEMA}.gold_provider_quality"

#  Read tables
df_bronze = spark.read.table(BRONZE_TABLE)
df_quarantine = spark.read.table(QUARANTINE_TABLE)

#  Total records per provider
df_total = df_bronze.groupBy("data_provider").agg(
    count("*").alias("total_records")
)

# Quarantined records per provider
df_quar = df_quarantine.groupBy("data_provider").agg(
    count("*").alias("quarantined_records")
)

#  Duplicate records (based on same city + event_time)
df_dup = df_bronze.groupBy("data_provider", "city", "event_time") \
    .count() \
    .filter(col("count") > 1)

df_dup = df_dup.groupBy("data_provider").agg(
    count("*").alias("duplicate_records")
)

#  Join all metrics
df_joined = df_total \
    .join(df_quar, "data_provider", "left") \
    .join(df_dup, "data_provider", "left")

# Fill nulls (important)
df_joined = df_joined.fillna({
    "quarantined_records": 0,
    "duplicate_records": 0
})

#Calculate quality_score
df_final = df_joined.withColumn(
    "quality_score",
    spark_round(
        100 - (
            (col("quarantined_records") + col("duplicate_records")) 
            / col("total_records") * 100
        ),
        2
    )
)

# Select final columns
df_gold_provider = df_final.select(
    "data_provider",
    "total_records",
    "quarantined_records",
    "duplicate_records",
    "quality_score"
).orderBy(col("quality_score").asc())

#  Write table
df_gold_provider.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(GOLD_PROVIDER_TABLE)

print(" Provider quality table created successfully!")

📊 Provider quality table created successfully!


In [0]:
spark.read.table(GOLD_PROVIDER_TABLE).display()

data_provider,total_records,quarantined_records,duplicate_records,quality_score
SkyWatch,340,115,42,53.82
AtmosAPI,404,26,39,83.91
ClimaData,420,7,37,89.52
WeatherPro,456,7,32,91.45
